In [ ]:
# %% [markdown]
# # 📊 FinCoreLite — Telegram-бот для Jupyter Notebook
# 
# Запустите эту ячейку для установки зависимостей:

# %%
# Установка необходимых библиотек
import sys
#!{sys.executable} -m pip install aiogram aiogram-calendar pandas moexalgo openpyxl nest_asyncio ipywidgets IPython --upgrade

# %% [markdown]
# # Импорт библиотек и настройка

# %%
import os
import asyncio
import logging
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import nest_asyncio
from IPython.display import display, HTML
import re

# Применяем nest_asyncio для работы в Jupyter
nest_asyncio.apply()

from aiogram import Bot, Dispatcher, types, F
from aiogram.filters import Command
from aiogram.fsm.context import FSMContext
from aiogram.fsm.state import State, StatesGroup
from aiogram.fsm.storage.memory import MemoryStorage
from aiogram.types import (FSInputFile, ReplyKeyboardMarkup, KeyboardButton,
                           InlineKeyboardMarkup, InlineKeyboardButton, CallbackQuery)
from aiogram_calendar import DialogCalendar, DialogCalendarCallback
from moexalgo import Ticker

# %% [markdown]
# # Настройка логирования и токена

# %%
# Настройка логирования для Jupyter
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('bot.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Токен бота (работает сразу, без ввода)
TOKEN = '8378361776:AAEvrhR8rQ-6zHgQ-6un4OBpznb6M7qKqBE'

# %% [markdown]
# # Классы и функции бота

# %%
# --- Состояния FSM ---
class StockState(StatesGroup):
    choosing_ticker = State()          # Выбор тикера через карусель
    searching_ticker = State()          # Поиск тикера
    choosing_metrics = State()         # Выбор метрик для отображения
    choosing_start_date = State()      # Выбор даты начала
    choosing_end_date = State()        # Выбор даты окончания
    choosing_format = State()           # Выбор формата выгрузки

# --- Константы ---
METRICS = {
    'open': '📊 Цена открытия',
    'high': '📈 Максимальная цена',
    'low': '📉 Минимальная цена',
    'close': '🎯 Цена закрытия',
    'avg': '⚖️ Средняя цена',
    'value': '💰 Оборот (руб)'
}

# --- Маппинг для названий колонок в Excel ---
EXCEL_COLUMN_NAMES = {
    'date': 'Дата',
    'open': 'Цена открытия',
    'high': 'Макс. цена',
    'low': 'Мин. цена',
    'close': 'Цена закрытия',
    'avg': 'Средняя цена',
    'value': 'Оборот (руб)'
}

# --- Класс для карусели тикеров (скроллящаяся лента) ---
class TickerCarousel:
    def __init__(self, tickers, page_size=1):
        self.tickers = tickers
        self.page_size = page_size
        self.total_pages = len(tickers)
    
    def get_current_ticker(self, index):
        """Получить тикер по индексу"""
        if 0 <= index < len(self.tickers):
            return self.tickers[index]
        return None
    
    def get_keyboard(self, current_index=0):
        """Создать инлайн клавиатуру с каруселью"""
        current_ticker = self.get_current_ticker(current_index)
        
        # Кнопки навигации (влево/вправо)
        nav_buttons = []
        
        # Кнопка "Назад" (◀️)
        if current_index > 0:
            nav_buttons.append(
                InlineKeyboardButton(text="◀️", callback_data=f"ticker_prev_{current_index-1}")
            )
        else:
            nav_buttons.append(
                InlineKeyboardButton(text="⏺️", callback_data="noop")
            )
        
        # Текущий тикер
        nav_buttons.append(
            InlineKeyboardButton(
                text=f"📈 {current_ticker} 📈", 
                callback_data=f"ticker_select_{current_ticker}"
            )
        )
        
        # Кнопка "Вперед" (▶️)
        if current_index < self.total_pages - 1:
            nav_buttons.append(
                InlineKeyboardButton(text="▶️", callback_data=f"ticker_next_{current_index+1}")
            )
        else:
            nav_buttons.append(
                InlineKeyboardButton(text="⏺️", callback_data="noop")
            )
        
        # Информация о позиции
        position_info = [
            InlineKeyboardButton(
                text=f"📄 {current_index + 1} из {self.total_pages}", 
                callback_data="noop"
            )
        ]
        
        # Кнопка выбора текущего тикера
        select_button = [
            InlineKeyboardButton(
                text=f"✅ Выбрать {current_ticker}", 
                callback_data=f"ticker_confirm_{current_ticker}"
            )
        ]
        
        # Кнопки поиска и меню
        action_buttons = [
            InlineKeyboardButton(text="🔍 Поиск тикера", callback_data="search_ticker"),
            InlineKeyboardButton(text="🏠 Меню", callback_data="restart")
        ]
        
        # Собираем клавиатуру
        keyboard = [nav_buttons, position_info, select_button, action_buttons]
        
        return InlineKeyboardMarkup(inline_keyboard=keyboard)

# --- Класс для выбора метрик ---
class MetricsSelector:
    def __init__(self, all_metrics):
        self.all_metrics = all_metrics
    
    def get_keyboard(self, selected_metrics=None):
        """Создать клавиатуру для выбора метрик"""
        if selected_metrics is None:
            selected_metrics = []
        
        buttons = []
        for metric_key, metric_name in self.all_metrics.items():
            emoji = "✅ " if metric_key in selected_metrics else "⬜ "
            buttons.append([
                InlineKeyboardButton(
                    text=f"{emoji}{metric_name}", 
                    callback_data=f"metric_toggle_{metric_key}"
                )
            ])
        
        # Кнопки управления
        control_buttons = [
            [
                InlineKeyboardButton(text="✅ Выбрать все", callback_data="metric_all"),
                InlineKeyboardButton(text="❌ Очистить все", callback_data="metric_none")
            ],
            [
                InlineKeyboardButton(text="📊 Показать данные", callback_data="metric_confirm")
            ],
            [
                InlineKeyboardButton(text="🏠 Меню", callback_data="restart")
            ]
        ]
        
        buttons.extend(control_buttons)
        return InlineKeyboardMarkup(inline_keyboard=buttons)

# --- Функция для поиска тикеров ---
def search_tickers(query, tickers, limit=10):
    """Поиск тикеров по запросу (регистронезависимый)"""
    if not query:
        return []
    
    query = query.upper().strip()
    # Сначала ищем точные совпадения в начале
    exact_matches = [t for t in tickers if t.startswith(query)]
    # Затем ищем частичные совпадения
    partial_matches = [t for t in tickers if query in t and t not in exact_matches]
    
    # Объединяем и ограничиваем количество
    results = (exact_matches + partial_matches)[:limit]
    return results

# --- Путь к файлу тикеров ---
TICKERS_PATH = Path(r"https://github.com/maksim200108-bit/bot/blob/main/Tickers.xlsx")
DEFAULT_TICKERS = [
    'SBER', 'GAZP', 'LKOH', 'NVTK', 'ROSN',
    'YNDX', 'MAIL', 'TCSG', 'SBERP', 'GMKN', 'NLMK', 'MAGN', 'CHMF',
    'AFLT', 'VTBR', 'MOEX', 'TATN', 'SNGS'
]

# --- Функции для работы с тикерами ---
def get_tickers():
    """Получить список тикеров из Excel; при отсутствии файла — список по умолчанию."""
    if TICKERS_PATH.exists():
        try:
            df = pd.read_excel(TICKERS_PATH)
            col = "Ticker" if "Ticker" in df.columns else df.columns[0]
            return df[col].astype(str).str.strip().str.upper().tolist()
        except Exception as e:
            logger.warning("Не удалось прочитать тикеры из файла: %s", e)
    return DEFAULT_TICKERS.copy()

def format_data_for_display(df, selected_metrics, ticker, start_date, end_date):
    """Форматировать данные для отображения в сообщении"""
    if df.empty:
        return "❌ Нет данных за выбранный период"
    
    # Основная статистика
    start_price = round(df.iloc[0]['close'], 2)
    end_price = round(df.iloc[-1]['close'], 2)
    change = round(((end_price - start_price) / start_price) * 100, 2)
    
    # Явно указываем знак минус для отрицательных изменений
    if change >= 0:
        change_str = f"+{change}%"
        trend = "📈"
    else:
        change_str = f"{change}%"
        trend = "📉"
    
    # Формируем заголовок
    header = [
        f"<b>📊 {ticker}</b>",
        f"📅 Период: {start_date} - {end_date}",
        f"💰 Изменение цены: {trend} {change_str} ({start_price} → {end_price})",
        "━" * 20
    ]
    
    # Добавляем выбранные метрики
    metrics_lines = []
    for metric in selected_metrics:
        if metric in df.columns:
            if metric in ['open', 'high', 'low', 'close', 'avg']:
                min_val = round(df[metric].min(), 2)
                max_val = round(df[metric].max(), 2)
                avg_val = round(df[metric].mean(), 2)
                metrics_lines.append(
                    f"<b>{METRICS[metric]}</b>\n"
                    f"  📊 Средняя: {avg_val}\n"
                    f"  📈 Макс: {max_val}\n"
                    f"  📉 Мин: {min_val}"
                )
            elif metric == 'value':
                total = df[metric].sum()
                avg_val = round(df[metric].mean(), 0)
                total = f"{total:,.0f} ₽"
                avg_val = f"{avg_val:,.0f} ₽"
                
                metrics_lines.append(
                    f"<b>{METRICS[metric]}</b>\n"
                    f"  📊 Средний: {avg_val}\n"
                    f"  📈 Всего: {total}"
                )
    
    return "\n\n".join(header + metrics_lines)

# --- Создание бота и регистрация обработчиков ---
bot = Bot(token=TOKEN)
dp = Dispatcher(storage=MemoryStorage())

def setup_handlers():
    """Регистрация обработчиков бота"""
    
    @dp.message(Command("start"))
    @dp.callback_query(F.data == "restart")
    async def cmd_start(message: types.Message | CallbackQuery, state: FSMContext):
        await state.clear()
        
        if isinstance(message, CallbackQuery):
            await message.answer()
            await message.message.delete()
            target = message.message
        else:
            target = message
        
        # Обновленное описание бота
        kb = InlineKeyboardMarkup(inline_keyboard=[
            [InlineKeyboardButton(text="📊 Выбрать акции и выгрузить данные", callback_data="start_carousel")],
            [InlineKeyboardButton(text="🔄 Перезапустить бота", callback_data="hard_restart")]
        ])
        
        await target.answer(
            "FinCoreLite — Telegram-бот для инвесторов и аналитиков.\n\n"
            "📊FinCoreLite — что умеет бот:\n"
            "• выгружать рыночные котировки\n"
            "• рассчитывать динамику акции",
            parse_mode="HTML",
            reply_markup=kb
        )
    
    @dp.callback_query(F.data == "hard_restart")
    async def hard_restart(callback: CallbackQuery, state: FSMContext):
        """Полный перезапуск бота"""
        await state.clear()
        await callback.message.delete()
        
        kb = InlineKeyboardMarkup(inline_keyboard=[
            [InlineKeyboardButton(text="📊 Выбрать акции и выгрузить данные", callback_data="start_carousel")],
            [InlineKeyboardButton(text="🔄 Перезапустить бота", callback_data="hard_restart")]
        ])
        
        await callback.message.answer(
            "🔄 <b>Бот перезапущен</b>\n\n"
            "FinCoreLite — Telegram-бот для инвесторов и аналитиков.\n\n"
            "📊FinCoreLite — что умеет бот:\n"
            "• выгружать рыночные котировки\n"
            "• рассчитывать динамику акции",
            parse_mode="HTML",
            reply_markup=kb
        )
    
    @dp.callback_query(F.data == "start_carousel")
    async def start_ticker_carousel(callback: CallbackQuery, state: FSMContext):
        tickers = get_tickers()
        carousel = TickerCarousel(tickers, page_size=1)
        
        await state.update_data(carousel=carousel, current_index=0)
        
        await callback.message.edit_text(
            "📈 <b>Выберите тикер:</b>\n"
            "Используйте кнопки ◀️ ▶️ для навигации или 🔍 для поиска",
            parse_mode="HTML",
            reply_markup=carousel.get_keyboard(current_index=0)
        )
        await state.set_state(StockState.choosing_ticker)
    
    @dp.callback_query(F.data == "search_ticker", StockState.choosing_ticker)
    async def search_ticker_start(callback: CallbackQuery, state: FSMContext):
        await callback.message.edit_text(
            "🔍 <b>Поиск тикера</b>\n\n"
            "Введите название тикера (например: SBER, GAZP, YNDX):\n"
            "Поиск работает без учета регистра",
            parse_mode="HTML",
            reply_markup=InlineKeyboardMarkup(inline_keyboard=[
                [InlineKeyboardButton(text="◀️ Назад к выбору", callback_data="start_carousel")],
                [InlineKeyboardButton(text="🏠 Меню", callback_data="restart")]
            ])
        )
        await state.set_state(StockState.searching_ticker)
    
    @dp.message(StockState.searching_ticker)
    async def process_ticker_search(message: types.Message, state: FSMContext):
        query = message.text.strip()
        tickers = get_tickers()
        
        # Выполняем поиск
        results = search_tickers(query, tickers)
        
        if not results:
            await message.answer(
                f"❌ По запросу '{query}' ничего не найдено.\n\n"
                "Попробуйте другой запрос или вернитесь к выбору:",
                reply_markup=InlineKeyboardMarkup(inline_keyboard=[
                    [InlineKeyboardButton(text="◀️ Назад к выбору", callback_data="start_carousel")],
                    [InlineKeyboardButton(text="🏠 Меню", callback_data="restart")]
                ])
            )
            return
        
        # Создаем кнопки с результатами поиска
        result_buttons = []
        for ticker in results:
            result_buttons.append([
                InlineKeyboardButton(
                    text=f"📈 {ticker}", 
                    callback_data=f"search_confirm_{ticker}"
                )
            ])
        
        # Добавляем кнопки навигации
        result_buttons.append([
            InlineKeyboardButton(text="◀️ Новый поиск", callback_data="search_ticker"),
            InlineKeyboardButton(text="🏠 Меню", callback_data="restart")
        ])
        
        await message.answer(
            f"🔍 Найдено тикеров: {len(results)}\n\n"
            "Выберите подходящий:",
            reply_markup=InlineKeyboardMarkup(inline_keyboard=result_buttons)
        )
    
    # Обработчик для подтверждения тикера из поиска
    @dp.callback_query(lambda c: c.data.startswith('search_confirm_'))
    async def confirm_search_ticker(callback: CallbackQuery, state: FSMContext):
        ticker = callback.data.replace('search_confirm_', '')
        await state.update_data(ticker=ticker)
        
        # Переходим к выбору метрик
        selector = MetricsSelector(METRICS)
        await callback.message.edit_text(
            f"✅ Выбран тикер: <b>{ticker}</b>\n\n"
            "📊 <b>Выберите метрики для анализа:</b>\n"
            "Нажимайте на метрики, чтобы выбрать/отменить",
            parse_mode="HTML",
            reply_markup=selector.get_keyboard(selected_metrics=['open', 'high', 'low', 'close'])
        )
        await state.set_state(StockState.choosing_metrics)
    
    @dp.callback_query(lambda c: c.data.startswith('ticker_prev_') or c.data.startswith('ticker_next_'), StockState.choosing_ticker)
    async def navigate_tickers(callback: CallbackQuery, state: FSMContext):
        if callback.data.startswith('ticker_prev_'):
            new_index = int(callback.data.split('_')[2])
        else:  # ticker_next_
            new_index = int(callback.data.split('_')[2])
        
        data = await state.get_data()
        carousel = data.get('carousel')
        
        await callback.message.edit_reply_markup(
            reply_markup=carousel.get_keyboard(current_index=new_index)
        )
        await state.update_data(current_index=new_index)
    
    @dp.callback_query(lambda c: c.data.startswith('ticker_select_'), StockState.choosing_ticker)
    async def select_ticker(callback: CallbackQuery, state: FSMContext):
        ticker = callback.data.replace('ticker_select_', '')
        await state.update_data(selected_ticker=ticker)
        await callback.answer(f"Выбран тикер {ticker}")
    
    @dp.callback_query(lambda c: c.data.startswith('ticker_confirm_'), StockState.choosing_ticker)
    async def confirm_ticker(callback: CallbackQuery, state: FSMContext):
        ticker = callback.data.replace('ticker_confirm_', '')
        await state.update_data(ticker=ticker)
        
        selector = MetricsSelector(METRICS)
        await callback.message.edit_text(
            f"✅ Выбран тикер: <b>{ticker}</b>\n\n"
            "📊 <b>Выберите метрики для анализа:</b>\n"
            "Нажимайте на метрики, чтобы выбрать/отменить",
            parse_mode="HTML",
            reply_markup=selector.get_keyboard(selected_metrics=['open', 'high', 'low', 'close'])
        )
        await state.set_state(StockState.choosing_metrics)
    
    @dp.callback_query(lambda c: c.data.startswith('metric_'), StockState.choosing_metrics)
    async def handle_metrics_selection(callback: CallbackQuery, state: FSMContext):
        data = await state.get_data()
        selected_metrics = data.get('selected_metrics', ['open', 'high', 'low', 'close']).copy()
        
        if callback.data.startswith('metric_toggle_'):
            metric = callback.data.replace('metric_toggle_', '')
            if metric in selected_metrics:
                selected_metrics.remove(metric)
            else:
                selected_metrics.append(metric)
        elif callback.data == 'metric_all':
            selected_metrics = list(METRICS.keys())
        elif callback.data == 'metric_none':
            selected_metrics = []
        elif callback.data == 'metric_confirm':
            if not selected_metrics:
                await callback.answer("❌ Выберите хотя бы одну метрику!", show_alert=True)
                return
            
            await state.update_data(selected_metrics=selected_metrics)
            
            await callback.message.edit_text(
                f"✅ Выбрано метрик: {len(selected_metrics)}\n\n"
                "📅 <b>Выберите дату начала:</b>",
                parse_mode="HTML",
                reply_markup=await DialogCalendar().start_calendar()
            )
            await state.set_state(StockState.choosing_start_date)
            return
        
        await state.update_data(selected_metrics=selected_metrics)
        selector = MetricsSelector(METRICS)
        await callback.message.edit_reply_markup(
            reply_markup=selector.get_keyboard(selected_metrics)
        )
    
    @dp.callback_query(DialogCalendarCallback.filter(), StockState.choosing_start_date)
    async def process_start_date(callback: CallbackQuery, callback_data: dict, state: FSMContext):
        selected, date = await DialogCalendar().process_selection(callback, callback_data)
        if selected:
            await state.update_data(start_date=date.strftime("%Y-%m-%d"))
            await callback.message.edit_text(
                f"✅ Начало: <b>{date.strftime('%d.%m.%Y')}</b>\n\n"
                "📅 <b>Выберите дату окончания:</b>",
                parse_mode="HTML",
                reply_markup=await DialogCalendar().start_calendar()
            )
            await state.set_state(StockState.choosing_end_date)
    
    @dp.callback_query(DialogCalendarCallback.filter(), StockState.choosing_end_date)
    async def process_end_date(callback: CallbackQuery, callback_data: dict, state: FSMContext):
        selected, date = await DialogCalendar().process_selection(callback, callback_data)
        if selected:
            data = await state.get_data()
            end_date_str = date.strftime("%Y-%m-%d")
            start_date_str = data.get('start_date')
            
            # Проверяем, что дата окончания не раньше даты начала
            if end_date_str < start_date_str:
                # Показываем предупреждение и возвращаем к выбору даты окончания
                await callback.answer("❌ Дата окончания не может быть раньше даты начала! Выберите другую дату.", show_alert=True)
                
                # Возвращаем пользователя к выбору даты окончания
                await callback.message.edit_text(
                    f"❌ Ошибка: дата окончания <b>{date.strftime('%d.%m.%Y')}</b> раньше даты начала <b>{datetime.strptime(start_date_str, '%Y-%m-%d').strftime('%d.%m.%Y')}</b>\n\n"
                    "📅 <b>Выберите дату окончания заново:</b>\n"
                    "Дата окончания должна быть позже даты начала",
                    parse_mode="HTML",
                    reply_markup=await DialogCalendar().start_calendar()
                )
                # Оставляем состояние выбора даты окончания
                return
            
            # Если дата корректная, сохраняем и идем дальше
            await state.update_data(end_date=end_date_str)
            
            kb = InlineKeyboardMarkup(inline_keyboard=[
                [
                    InlineKeyboardButton(text="📗 Excel", callback_data="format_excel"),
                    InlineKeyboardButton(text="📋 CSV", callback_data="format_csv")
                ],
                [
                    InlineKeyboardButton(text="🏠 Меню", callback_data="restart")
                ]
            ])
            
            summary = (
                f"<b>📊 Параметры анализа:</b>\n\n"
                f"Тикер: <b>{data['ticker']}</b>\n"
                f"Метрики: <b>{', '.join([METRICS[m] for m in data['selected_metrics']])}</b>\n"
                f"Период: <b>{data['start_date']} - {end_date_str}</b>\n\n"
                f"💾 Выберите формат выгрузки:"
            )
            
            await callback.message.edit_text(summary, parse_mode="HTML", reply_markup=kb)
            await state.set_state(StockState.choosing_format)
    
    @dp.callback_query(lambda c: c.data.startswith('format_'), StockState.choosing_format)
    async def process_format_and_download(callback: CallbackQuery, state: FSMContext):
        format_type = callback.data.replace('format_', '')
        data = await state.get_data()
        
        await callback.message.edit_text("⚡ <b>Загружаем данные с MOEX...</b>", parse_mode="HTML")
        
        try:
            ticker = Ticker(data['ticker'])
            df = ticker.candles(start=data['start_date'], end=data['end_date'], period='1d')
            
            if df.empty:
                await callback.message.edit_text(
                    "❌ Нет данных за выбранный период.\n\n"
                    "Попробуйте другой период или тикер.",
                    reply_markup=InlineKeyboardMarkup(inline_keyboard=[
                        [InlineKeyboardButton(text="🏠 Меню", callback_data="restart")]
                    ])
                )
                await state.clear()
                return
            
            df['avg'] = df.apply(
                lambda row: round(row['value'] / row['volume'], 2) if row['volume'] != 0 else row['close'],
                axis=1
            )
            
            display_text = format_data_for_display(
                df, data['selected_metrics'], data['ticker'],
                data['start_date'], data['end_date']
            )
            
            filename = f"MOEX_{data['ticker']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            
            columns_to_save = ['begin'] + data['selected_metrics']
            df_to_save = df[columns_to_save].copy()
            df_to_save.rename(columns={'begin': 'date'}, inplace=True)
            
            # Переименовываем колонки для Excel
            df_to_save.rename(columns=EXCEL_COLUMN_NAMES, inplace=True)
            
            # Сохранение в выбранном формате
            if format_type == 'excel':
                excel_filename = f"{filename}.xlsx"
                df_to_save.to_excel(excel_filename, index=False)
                send_filename = excel_filename
            else:
                csv_filename = f"{filename}.csv"
                df_to_save.to_csv(csv_filename, index=False, encoding='utf-8-sig')
                send_filename = csv_filename
            
            await bot.send_document(
                callback.from_user.id,
                FSInputFile(send_filename),
                caption=display_text[:900],
                parse_mode="HTML"
            )
            
            # Удаляем файл после отправки
            os.remove(send_filename)
            
            await callback.message.delete()
            await bot.send_message(
                callback.from_user.id,
                "✅ <b>Данные успешно выгружены!</b>\n\n"
                "Что делаем дальше?",
                parse_mode="HTML",
                reply_markup=InlineKeyboardMarkup(inline_keyboard=[
                    [InlineKeyboardButton(text="📊 Другой тикер", callback_data="start_carousel")],
                    [InlineKeyboardButton(text="🏠 Меню", callback_data="restart")]
                ])
            )
            
        except Exception as e:
            logger.error(f"Ошибка при выгрузке: {e}")
            error_message = str(e)
            await callback.message.edit_text(
                f"❌ Произошла ошибка: {error_message[:200]}\n\n"
                "Попробуйте позже или выберите другой тикер.",
                reply_markup=InlineKeyboardMarkup(inline_keyboard=[
                    [InlineKeyboardButton(text="🏠 Меню", callback_data="restart")]
                ])
            )
        finally:
            await state.clear()

setup_handlers()

# %% [markdown]
# # Запуск бота

# %%
logger.info("Бот FinCoreLite запущен")
nest_asyncio.apply()
asyncio.run(dp.start_polling(bot, skip_updates=True))

# %% [markdown]
# # Полезные функции для Jupyter

# %%
def show_tickers_table():
    """Показать таблицу с тикерами прямо в Jupyter"""
    tickers = get_tickers()
    
    # Разбиваем на колонки для красивого отображения
    cols = 5
    rows = [tickers[i:i+cols] for i in range(0, len(tickers), cols)]
    
    html = "<h3>📋 Доступные тикеры MOEX</h3>"
    html += "<table style='border-collapse: collapse;'>"
    
    for row in rows:
        html += "<tr>"
        for ticker in row:
            html += f"<td style='padding: 8px; border: 1px solid #ddd;'><b>{ticker}</b></td>"
        # Добавляем пустые ячейки, если строка неполная
        for _ in range(cols - len(row)):
            html += "<td style='padding: 8px; border: 1px solid #ddd;'></td>"
        html += "</tr>"
    
    html += "</table>"
    display(HTML(html))

def test_ticker_data(ticker='SBER', days=30):
    """Тестовая функция для проверки получения данных прямо в Jupyter"""
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    print(f"📊 Тестовый запрос для {ticker} за последние {days} дней...")
    
    try:
        t = Ticker(ticker)
        df = t.candles(start=start_date.strftime('%Y-%m-%d'), 
                       end=end_date.strftime('%Y-%m-%d'), 
                       period='1d')
        
        if not df.empty:
            df['avg'] = df.apply(
                lambda row: round(row['value'] / row['volume'], 2) if row['volume'] != 0 else row['close'],
                axis=1
            )
            
            print(f"✅ Получено {len(df)} записей")
            display(df[['begin', 'open', 'high', 'low', 'close', 'value', 'avg']].head())
            
            # Базовая статистика
            print(f"\n📈 Статистика:")
            print(f"  Цена на начало: {df.iloc[0]['close']:.2f}")
            print(f"  Цена на конец: {df.iloc[-1]['close']:.2f}")
            
            change = ((df.iloc[-1]['close'] - df.iloc[0]['close']) / df.iloc[0]['close'] * 100)
            if change >= 0:
                print(f"  Изменение цены: +{change:.2f}%")
            else:
                print(f"  Изменение цены: {change:.2f}%")
        else:
            print("❌ Нет данных")
    except Exception as e:
        print(f"❌ Ошибка: {e}")

# %% [markdown]
# # Инструкция
# 
# 1. **Выполните все ячейки** сверху вниз (Run All) — бот запустится автоматически.
# 2. Откройте бота в Telegram и отправьте `/start`.
# 3. Остановка: прервите выполнение ячейки (Stop / Interrupt kernel).
# 
# **В Telegram:** `/start` → выбор тикера (скроллинг или поиск) → выбор метрик → выбор дат → выгрузка в Excel/CSV.


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


2026-02-17 23:09:04,086 - __main__ - INFO - Бот FinCoreLite запущен
2026-02-17 23:09:04,088 - aiogram.dispatcher - INFO - Start polling
2026-02-17 23:09:04,789 - aiogram.dispatcher - INFO - Run polling for bot @fincorelitebot id=8378361776 - 'FinCore (Lite)'
2026-02-17 23:09:09,507 - aiogram.event - INFO - Update id=924223674 is handled. Duration 765 ms by bot id=8378361776
2026-02-17 23:09:12,289 - aiogram.event - INFO - Update id=924223675 is handled. Duration 405 ms by bot id=8378361776
2026-02-17 23:09:20,825 - aiogram.event - INFO - Update id=924223676 is handled. Duration 563 ms by bot id=8378361776
2026-02-17 23:09:22,211 - aiogram.event - INFO - Update id=924223677 is handled. Duration 561 ms by bot id=8378361776
2026-02-17 23:09:22,747 - aiogram.event - INFO - Update id=924223678 is handled. Duration 78 ms by bot id=8378361776
2026-02-17 23:09:25,111 - aiogram.event - INFO - Update id=924223679 is handled. Duration 94 ms by bot id=8378361776
2026-02-17 23:09:27,689 - aiogram.e